# Missão Aurora Siger — Relatório Operacional de Pré-Decolagem

**Disciplina integradora:** Ciência da Computação, Pensamento Computacional (Python), IA & Prompt Engineering, Energias Renováveis e Sustentáveis, Formação Social e Sustentabilidade

**Grupo 18:** Felipe de Barros Franco e Guilherme Santos

**Objetivo:** reunir as verificações técnicas, energéticas e éticas usadas para liberar ou abortar a decolagem da nave Aurora Siger.

O notebook segue a mesma ordem dos itens pedidos na atividade:

1. [1.1 Organização e descrição da telemetria](#1.1)
2. [1.2 Algoritmo de verificação](#1.2)
3. [1.3 Script em Python](#1.3)
4. [1.4 Análise energética](#1.4)
5. [1.5 Análise assistida por IA](#1.5)
6. [1.6 Reflexão crítica](#1.6)


<a id="1.1"></a>
## 1.1 Organização e descrição da telemetria

Durante a janela de pré-decolagem, os sensores da nave enviam um conjunto fixo de leituras. A tabela abaixo lista o que cada leitura mede, em que unidade ela chega e qual a faixa considerada segura. Essas faixas vêm dos parâmetros de engenharia da missão e são as mesmas usadas pelo algoritmo do item 1.2 como critério de decisão.

| Parâmetro | Unidade | Faixa segura | Descrição |
|---|---|---|---|
| Temperatura interna | °C | 18 a 27 | Temperatura da cabine e da eletrônica embarcada. Fora dessa faixa há risco térmico para os sistemas e para a tripulação. |
| Temperatura externa | °C | -60 a 45 | Temperatura do casco e do ambiente externo, acompanhada para avaliar estresse térmico na estrutura. |
| Integridade estrutural | booleano (0/1) | deve ser 1 | Indica se o casco e as junções estão íntegros (1) ou comprometidos (0). |
| Nível de energia | % | mínimo 40% | Carga atual da bateria principal em relação à capacidade total. |
| Pressão dos tanques (oxidante e combustível) | bar | 150 a 200 | Pressurização dos propelentes. Abaixo da faixa há risco de falha de queima; acima, risco de sobrepressão. |
| Status dos módulos críticos | booleano por módulo (0/1) | todos = 1 | Navegação, comunicação, propulsão, suporte de vida e aviônica. Cada módulo precisa reportar 1 (operacional). |

No script, essas leituras são agrupadas em um dicionário que representa um instante de telemetria. Foram montados dois cenários: um nominal, com todos os valores dentro da faixa, e um com falhas. O segundo serve para mostrar que o algoritmo aborta a decolagem quando algo sai do esperado, e não apenas confirma o caso favorável.


<a id="1.2"></a>
## 1.2 Algoritmo de verificação

A verificação é uma sequência de comparações entre o valor lido e a faixa segura correspondente. O algoritmo não para na primeira irregularidade: ele testa todos os parâmetros e vai acumulando as não conformidades em uma lista. No final, lista vazia significa "PRONTO PARA DECOLAR" e qualquer item na lista significa "DECOLAGEM ABORTADA". Fazer assim custa o mesmo e entrega à equipe de missão, de uma só vez, tudo o que precisa ser corrigido antes de uma nova tentativa.

**Pseudocódigo:**

```
INÍCIO
LER temperatura_interna, temperatura_externa, integridade_estrutural,
    nivel_energia, pressao_tanque_oxidante, pressao_tanque_combustivel,
    modulos_criticos[]

falhas <- lista vazia

SE temperatura_interna NÃO ESTÁ ENTRE 18 E 27 ENTÃO
    ADICIONAR falha "temperatura interna fora da faixa"

SE temperatura_externa NÃO ESTÁ ENTRE -60 E 45 ENTÃO
    ADICIONAR falha "temperatura externa fora da faixa"

SE integridade_estrutural != 1 ENTÃO
    ADICIONAR falha "integridade estrutural comprometida"

SE nivel_energia < 40 ENTÃO
    ADICIONAR falha "energia insuficiente"

PARA CADA tanque EM [oxidante, combustivel]
    SE pressao(tanque) NÃO ESTÁ ENTRE 150 E 200 ENTÃO
        ADICIONAR falha "pressão do tanque fora da faixa"

PARA CADA modulo EM modulos_criticos
    SE modulo != 1 ENTÃO
        ADICIONAR falha "módulo crítico com falha: " + nome_modulo

SE falhas está vazia ENTÃO
    status <- "PRONTO PARA DECOLAR"
SENÃO
    status <- "DECOLAGEM ABORTADA"

IMPRIMIR status
IMPRIMIR falhas (se houver)
FIM
```

**Fluxograma equivalente:**

![Fluxograma do algoritmo de verificação](imagens/fluxograma_verificacao.png)


<a id="1.3"></a>
## 1.3 Script em Python

O algoritmo foi implementado no arquivo `main.py`. Cada caixa do fluxograma virou uma função, e a `main` só chama essas funções na ordem. Assim o arquivo pode ser lido de cima para baixo sem que a lógica de verificação se misture com a de impressão.

| Função | O que faz |
|---|---|
| `gerar_telemetria_nominal()` | Devolve o dicionário do cenário sem falhas, com todos os valores dentro das faixas do item 1.1. |
| `gerar_telemetria_com_falha()` | Devolve o dicionário do cenário de teste com irregularidades, usado para exercitar o caminho de abort. |
| `verificar_decolagem(telemetria)` | Aplica as comparações do item 1.2. Percorre temperaturas, integridade, energia, pressão dos dois tanques e os cinco módulos críticos, acumulando as não conformidades em uma lista. Devolve o status e essa lista. |
| `imprimir_relatorio_verificacao(telemetria)` | Chama a verificação e formata a saída para leitura humana: timestamp, status final e, se houver, cada falha numerada com o valor medido e o esperado. |
| `calcular_autonomia(...)` | Executa as contas do item 1.4 e devolve os valores intermediários junto com a autonomia final. |

A separação entre `verificar_decolagem` e `imprimir_relatorio_verificacao` é o ponto principal do desenho. A primeira função só decide e não imprime nada, o que permite testá-la ou reaproveitá-la em outro contexto. A segunda cuida apenas da apresentação. A `main` instancia os dois cenários de telemetria e passa cada um pelo relatório.

Uma escolha herdada do item 1.2 aparece aqui na prática: a verificação não usa `return` no primeiro erro encontrado. Ela testa tudo e devolve a lista completa, por isso o cenário 2 mostra as quatro falhas de uma vez em vez de só a primeira.

### Saída do cenário 1 (telemetria nominal)

```
### CENÁRIO 1: TELEMETRIA NOMINAL ###

Timestamp da leitura: T-00:10:00
-------------------------------------------------------
RESULTADO: PRONTO PARA DECOLAR
Todos os parâmetros dentro das faixas seguras. Nave liberada.
=======================================================
```

### Saída do cenário 2 (telemetria com falhas)

```
### CENÁRIO 2: TELEMETRIA COM FALHAS ###

Timestamp da leitura: T-00:10:00
-------------------------------------------------------
RESULTADO: DECOLAGEM ABORTADA
Não conformidades encontradas:
  1. Temperatura interna fora da faixa segura: 29.8°C (esperado (18.0, 27.0))
  2. Nível de energia insuficiente: 33% (mínimo exigido: 40%)
  3. Pressão do tanque de combustivel fora da faixa segura: 141.0 bar (esperado (150.0, 200.0))
  4. Falha no módulo crítico: 'propulsao' (status=0)
=======================================================
```

As quatro falhas do cenário 2 são exatamente as que foram plantadas na telemetria de teste: temperatura interna 2,8 °C acima do limite, energia 7 pontos abaixo do mínimo, tanque de combustível despressurizado e módulo de propulsão fora do ar. Nenhuma passou despercebida e nenhum parâmetro correto foi acusado por engano.


<a id="1.4"></a>
## 1.4 Análise energética

O cálculo da autonomia inicial é feito em quatro passos. Primeiro converte-se a carga percentual em energia disponível, depois descontam-se as perdas, em seguida o consumo da própria decolagem, e o que sobra é dividido pelo consumo médio em cruzeiro:

$$E_{disponível} = C_{total} \times \dfrac{Carga\%}{100}$$

$$E_{útil} = E_{disponível} - (E_{disponível} \times \dfrac{Perdas\%}{100})$$

$$E_{pós\text{-}decolagem} = E_{útil} - Consumo_{decolagem}$$

$$Autonomia_{horas} = \dfrac{E_{pós\text{-}decolagem}}{Consumo_{cruzeiro}}$$

As variáveis de entrada são:

- **Capacidade total (kWh):** energia máxima que a bateria principal armazena;
- **Carga atual (%):** valor lido da telemetria (`nivel_energia_pct`);
- **Consumo estimado na decolagem (kWh):** energia gasta pelos sistemas durante a janela de decolagem;
- **Perdas energéticas (%):** dissipação térmica, conversão DC-DC e ineficiência dos inversores;
- **Consumo médio em cruzeiro (kW):** potência exigida pelos sistemas depois que a nave estabiliza.

Aplicando os valores da missão (bateria de 150 kWh, 87% de carga vindos da telemetria, 6% de perdas, 42 kWh gastos na decolagem e 3,2 kW médios em cruzeiro), a função de cálculo produz:

```
ANÁLISE ENERGÉTICA - AUTONOMIA INICIAL DA NAVE AURORA SIGER
-------------------------------------------------------
Capacidade total da bateria...........: 150.0 kWh
Carga atual (telemetria)..............: 87%
Energia disponível....................: 130.50 kWh
Perdas energéticas (6%)...............: 7.83 kWh
Energia útil (após perdas)............: 122.67 kWh
Consumo estimado na decolagem.........: 42.00 kWh
Energia restante pós-decolagem........: 80.67 kWh
Consumo médio em cruzeiro.............: 3.20 kW
AUTONOMIA ESTIMADA....................: 25.21 horas
=======================================================
```

![Análise energética da missão](imagens/analise_energetica.png)


**Leitura dos resultados.** Com 87% de carga em uma bateria de 150 kWh, a nave parte com 130,5 kWh. As perdas de 6% consomem 7,83 kWh e a decolagem em si consome 42 kWh, sobrando 80,67 kWh. Nos 3,2 kW médios do regime de cruzeiro, isso dá cerca de **25,21 horas** de operação.

Esse número é o que interessa para a equipe de missão: ele define quanto tempo a nave opera com segurança antes de precisar recorrer aos painéis solares ou entrar em modo de economia de energia. Vale notar que a carga de 87% é folgada em relação ao mínimo de 40%, mas o mínimo garante apenas que a decolagem é viável, não que a autonomia depois dela seja confortável.


<a id="1.5"></a>
## 1.5 Análise assistida por IA

A telemetria dos dois cenários foi submetida a um assistente de linguagem com o seguinte prompt:

> *"Classifique cada parâmetro de telemetria como NOMINAL, ATENÇÃO ou CRÍTICO em relação às faixas seguras informadas, aponte quaisquer anomalias entre os parâmetros (não apenas violações isoladas de faixa) e sugira o nível de risco geral da decolagem com uma recomendação objetiva."*

### Resposta da IA — Cenário 1 (telemetria nominal)

| Parâmetro | Valor | Classificação |
|---|---|---|
| Temperatura interna | 21,5 °C | NOMINAL |
| Temperatura externa | -42,3 °C | NOMINAL |
| Integridade estrutural | 1 | NOMINAL |
| Nível de energia | 87% | NOMINAL (margem confortável acima do mínimo de 40%) |
| Pressão tanque oxidante | 182,0 bar | NOMINAL |
| Pressão tanque combustível | 178,5 bar | NOMINAL |
| Módulos críticos | todos = 1 | NOMINAL |

*Anomalias identificadas:* nenhuma. A diferença de 3,5 bar entre os dois tanques fica dentro da variação esperada entre propelentes e não indica desbalanceamento.

*Sugestão de risco:* **BAIXO**. Recomendação: prosseguir com a contagem regressiva.

### Resposta da IA — Cenário 2 (telemetria com falhas)

| Parâmetro | Valor | Classificação |
|---|---|---|
| Temperatura interna | 29,8 °C | CRÍTICO (2,8 °C acima do limite de 27 °C) |
| Temperatura externa | -41,0 °C | NOMINAL |
| Integridade estrutural | 1 | NOMINAL |
| Nível de energia | 33% | CRÍTICO (7 p.p. abaixo do mínimo de 40%) |
| Pressão tanque oxidante | 182,0 bar | NOMINAL |
| Pressão tanque combustível | 141,0 bar | CRÍTICO (9 bar abaixo do mínimo de 150 bar) |
| Módulo "propulsão" | 0 | CRÍTICO (módulo inoperante) |

*Anomalias identificadas além das violações de faixa:*

1. **Temperatura alta junto com energia baixa.** A combinação sugere sobrecarga ou falha no sistema de refrigeração, que também puxa energia da bateria. Antes de uma nova tentativa, o subsistema térmico deveria ser inspecionado.
2. **Diferença de pressão entre os tanques.** Os 41 bar de diferença entre oxidante (182) e combustível (141) fogem muito do padrão de 3,5 bar visto no cenário nominal. Isso aponta mais para vazamento ou falha de pressurização no tanque de combustível do que para uma leitura pontual baixa.
3. **Módulo de propulsão inoperante.** Por si só já basta para abortar a decolagem. Como acontece junto com a queda de pressão do combustível, reforça a suspeita de um problema no sistema de propulsão como um todo.

*Sugestão de risco:* **ALTO**. Recomendação: abortar a decolagem, isolar o subsistema de propulsão e o tanque de combustível para inspeção e repetir a checagem completa antes da próxima janela de lançamento.

A IA entra aqui como camada de apoio à decisão, não como substituta do algoritmo dos itens 1.2 e 1.3. O algoritmo é determinístico e responde sempre a mesma coisa para a mesma entrada, o que é o que se espera de um sistema de segurança. A IA acrescenta uma leitura qualitativa, cruzando parâmetros que as regras de faixa avaliam isoladamente.


<a id="1.6"></a>
## 1.6 Reflexão crítica

### Ética e responsabilidade

Automatizar a decisão entre "PRONTO PARA DECOLAR" e "DECOLAGEM ABORTADA" coloca no código uma responsabilidade que, em uma missão real, envolve vidas e recursos de altíssimo valor. Duas consequências práticas vêm disso. A primeira é que as faixas seguras precisam ter origem em critério de engenharia, não em número escolhido por conveniência. A segunda é que o sistema tem de ser auditável: quem lê a saída precisa entender por que a nave foi liberada ou abortada. Foi por isso que o algoritmo registra cada não conformidade separadamente, em vez de devolver só um "sim" ou "não" sem justificativa.

A IA do item 1.5 exige o mesmo cuidado, por outro motivo. Modelos de linguagem encontram padrões úteis, mas também produzem conclusões inventadas com aparência de análise técnica. As sugestões deles servem para orientar a decisão humana; a palavra final continua sendo da verificação determinística.

### Impacto social da exploração espacial

Missões como a Aurora Siger dependem de investimento alto em ciência, engenharia e formação de pessoal especializado. Existe aí uma tensão legítima: parte desses recursos poderia ser aplicada em problemas terrestres imediatos, como saneamento, saúde e educação. Por outro lado, tecnologias nascidas da exploração espacial costumam voltar para a sociedade em usos civis, como sensores miniaturizados, sistemas de purificação de água, materiais mais leves e resistentes e algoritmos de decisão autônoma.

O ponto crítico não é escolher entre uma coisa e outra, mas quem fica com o resultado. Se o retorno dessa tecnologia se concentra em poucas empresas ou países, a exploração espacial aprofunda desigualdades de acesso que já existem em vez de reduzi-las.

### Sustentabilidade tecnológica

A análise do item 1.4 mostra em escala pequena algo que vale para qualquer sistema computacional: a energia disponível é finita e conhecida. Numa nave isso é evidente, porque a bateria tem capacidade fixa e não há como recarregar no meio do caminho. Em um data center na Terra a restrição é a mesma, só menos visível.

Escrever algoritmos eficientes, então, é mais do que boa prática de engenharia. Cada joule economizado em processamento é um joule que não precisou ser gerado, transmitido e armazenado, cada etapa com suas próprias perdas. O mesmo raciocínio se aplica ao projeto aeroespacial: geração renovável embarcada, reaproveitamento de componentes e controle de resíduos orbitais funcionam melhor como requisito desde o início do que como ajuste feito depois que o sistema já está pronto.


---
## Conclusão

O relatório operacional de pré-decolagem da Aurora Siger ficou completo: a telemetria foi organizada com faixas seguras justificadas, o algoritmo de verificação foi descrito em pseudocódigo e fluxograma e implementado em Python, os dois cenários de teste confirmaram os dois caminhos de decisão, a autonomia energética foi calculada em 25,21 horas e a análise assistida por IA levantou correlações que as regras de faixa não pegam sozinhas. A reflexão final trata dos limites éticos de automatizar esse tipo de decisão.

Instruções de execução no `README.md`.
